The Core Concept: Prefill vs. Decode  

Prefill Phase (Prompt Processing):Processes the entire initial prompt (e.g., $S$ tokens) in parallel.Computes Keys ($K$) and Values ($V$) for all prompt tokens at once.Stores these $K$ and $V$ tensors into a KV Cache.Outputs the logits for the very last token of the prompt to predict token $S+1$.

Decode Phase (Autoregressive Generation):Generates tokens one by one sequentially.For each new step, only the new single token is passed through the model.Its newly computed $K$ and $V$ vectors are appended to the existing KV Cache.This avoids recomputing attention over the entire historical sequence, reducing time complexity per step from $O(S^2)$ to $O(S)$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttentionWithKVCache(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.h_dim = d_model // n_heads

        self.qkv_linear = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.scale = 1.0 / (self.h_dim ** 0.5)

    def forward(self, x: torch.Tensor, kv_cache=None):
        """
        x: [B, S, D] (During prefill, S = prompt length. During decode, S = 1)
        kv_cache: Tuple of (past_k, past_v), each of shape [B, H, past_S, D_H]
        """
        B, S, D = x.shape

        # 1. Linear projection and chunking
        qkv = self.qkv_linear(x) # [B, S, 3*D]
        q, k, v = qkv.chunk(3, dim=-1)

        # 2. Reshape and permute to [B, H, S, D_H]
        q = q.view(B, S, self.n_heads, self.h_dim).transpose(1, 2)
        k = k.view(B, S, self.n_heads, self.h_dim).transpose(1, 2)
        v = v.view(B, S, self.n_heads, self.h_dim).transpose(1, 2)

        # 3. Handle KV Cache (Concatenate past K/V with current K/V)
        if kv_cache is not None:
            past_k, past_v = kv_cache
            k = torch.cat([past_k, k], dim=-2) # Concatenate along sequence dimension
            v = torch.cat([past_v, v], dim=-2)
        
        current_kv_cache = (k, v) # Save updated cache for next step

        # 4. Scaled Dot-Product Attention Scores: [B, H, S_q, S_total_k]
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        # 5. Apply Causal Mask (if prefill phase has multiple tokens)
        S_q = q.shape[-2]
        S_total_k = k.shape[-2]
        if S_q > 1:
            causal_mask = torch.triu(torch.full((S_q, S_total_k), float('-inf'), device=x.device), diagonal=S_total_k - S_q + 1)
            attn_scores = attn_scores + causal_mask

        attn_weights = F.softmax(attn_scores, dim=-1)
        out = torch.matmul(attn_weights, v) # [B, H, S_q, D_H]

        # 6. Reshape back and project
        out = out.transpose(1, 2).contiguous().view(B, S_q, D)
        return self.out_proj(out), current_kv_cache


class NextTokenPredictionEngine(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, n_heads: int):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.attention = CausalSelfAttentionWithKVCache(d_model, n_heads)
        self.ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def generate_next_token(self, input_ids: torch.Tensor, kv_cache=None):
        """
        Handles both Prefill (when kv_cache is None and input_ids length > 1) 
        and Decode (when input_ids length == 1 with an existing kv_cache).
        """
        # 1. Embed input tokens
        x = self.token_embedding(input_ids) # [B, S, D]

        # 2. Forward through attention with cache
        x_norm = self.ln(x)
        attn_out, new_kv_cache = self.attention(x_norm, kv_cache)
        
        # 3. Residual connection
        x = x + attn_out

        # 4. Predict logits for the LAST token in the sequence
        logits = self.lm_head(x[:, -1, :]) # [B, vocab_size]
        
        return logits, new_kv_cache

In [ ]:
import torch

# -------------------------------------------------------------
# 1. Setup Model Hyperparameters
# -------------------------------------------------------------
vocab_size = 1000  # Size of vocabulary
d_model = 64       # Hidden embedding dimension
n_heads = 4        # Number of attention heads
max_len = 512      # Maximum sequence length

# Instantiate our engine
model = NextTokenPredictionEngine(vocab_size=vocab_size, d_model=d_model, n_heads=n_heads)
model.eval() # Set to evaluation mode

# -------------------------------------------------------------
# 2. Define an Initial Prompt (Prefill Phase Input)
# -------------------------------------------------------------
# Let's say our prompt has 5 tokens: [12, 45, 89, 23, 7]
prompt_ids = torch.tensor([[12, 45, 89, 23, 7]], dtype=torch.long) # Shape: [Batch=1, Seq_Len=5]

print(f"--- PREFILL PHASE ---")
print(f"Input Prompt Token IDs: {prompt_ids.tolist()}")

# -------------------------------------------------------------
# 3. Call Prefill Phase
# -------------------------------------------------------------
with torch.no_grad():
    # Pass the entire prompt. kv_cache is initially None.
    logits, kv_cache = model.generate_next_token(prompt_ids, kv_cache=None)
    
    # Greedily pick the token with the highest logit score
    next_token_id = torch.argmax(logits, dim=-1, keepdim=True) # Shape: [1, 1]

print(f"Generated First Next Token ID: {next_token_id.item()}")

# Append the newly predicted token to our sequence tracking
generated_sequence = torch.cat([prompt_ids, next_token_id], dim=1)

# -------------------------------------------------------------
# 4. Call Decode Phase (Autoregressive Loop for N steps)
# -------------------------------------------------------------
print(f"\n--- DECODE PHASE (KV Caching Active) ---")
num_decode_steps = 5

for step in range(num_decode_steps):
    # Pass ONLY the single newest token, along with the existing kv_cache!
    # Shape of input token: [1, 1]
    logits, kv_cache = model.generate_next_token(next_token_id, kv_cache=kv_cache)
    
    # Predict the next token
    next_token_id = torch.argmax(logits, dim=-1, keepdim=True)
    
    # Append to sequence
    generated_sequence = torch.cat([generated_sequence, next_token_id], dim=1)
    
    print(f"Decode Step {step + 1}: Generated Token ID -> {next_token_id.item()}")

print(f"\nFinal Complete Generated Sequence IDs:")
print(generated_sequence.tolist())